# GEOCK + BindingDB: GPU-Accelerated Training
Downloads BindingDB (1M compounds) from HuggingFace, computes 2D fingerprints, trains XGBoost with GPU.

**Repo**: https://github.com/Hanishchow/gecokkkkkkkkq

In [ ]:
BASE = '/content/geock_bindingdb'
import os; os.makedirs(BASE, exist_ok=True)
%cd $BASE


In [ ]:
!pip install -q rdkit-pypi xgboost scikit-learn pandas pyarrow huggingface-hub scipy

import numpy as np, pandas as pd, pickle, time, os, json
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
import xgboost as xgb
import urllib.request

print(f'XGBoost: {xgb.__version__}')
!nvidia-smi | head -5

In [ ]:
# 1. Download BindingDB (~375 MB)
from huggingface_hub import hf_hub_download, login
import shutil

# Optional: set HF_TOKEN for faster downloads
# login(token='your_hf_token_here')

for i in range(2):
    f = f'data/train-0000{i}-of-00002.parquet'
    local = f'{BASE}/binddb_train_{i}.parquet'
    if os.path.exists(local):
        print(f'  train_{i}: exists ({os.path.getsize(local)/1e6:.0f} MB)')
        continue
    path = hf_hub_download('vladak/bindingdb', f, repo_type='dataset')
    shutil.copy2(path, local)
    print(f'  train_{i}: {os.path.getsize(local)/1e6:.0f} MB')

# Also download reference files from GitHub
REPO = 'https://raw.githubusercontent.com/Hanishchow/gecokkkkkkkkq/master/autoresearch_backup'
for f in ['casf2016_reference.csv', 'phase2_X.npy', 'phase2_y.npy', 'pocket_features_full.pkl', 'phase2_pdb_ids.pkl']:
    local = f'{BASE}/{f}'
    if not os.path.exists(local):
        urllib.request.urlretrieve(f'{REPO}/{f}', local)
        print(f'  {f}: {os.path.getsize(local)/1e6:.1f} MB')
    else:
        print(f'  {f}: exists ({os.path.getsize(local)/1e6:.1f} MB)')

In [ ]:
# 2. Compute 982-dim fingerprints in parallel
from multiprocessing import Pool

def compute_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    ecfp = AllChem.GetMorganFingerprintAsBitVect(mol, 4, nBits=512)
    maccs = MACCSkeys.GenMACCSKeys(mol)
    fcfp = AllChem.GetMorganFingerprintAsBitVect(mol, 4, nBits=300, useFeatures=True)
    fp = np.zeros(982, dtype=np.uint8)
    fp[:512] = np.array(ecfp)
    fp[512:679] = np.array(maccs)
    fp[679:979] = np.array(fcfp)
    return fp

def process_chunk(args):
    smiles_list, ic50_list = args
    fps, ys = [], []
    for smi, ic50 in zip(smiles_list, ic50_list):
        if np.isnan(ic50): continue
        fp = compute_fp(smi)
        if fp is not None:
            fps.append(fp); ys.append(ic50)
    return np.array(fps, dtype=np.uint8), np.array(ys, dtype=np.float64)

t0 = time.time()
dfs = [pd.read_parquet(f'{BASE}/binddb_train_{i}.parquet') for i in range(2)]
df_all = pd.concat(dfs, ignore_index=True)
print(f'{len(df_all)} entries loaded ({time.time()-t0:.0f}s)')

n_cores = os.cpu_count()
chunk_size = max(10000, len(df_all) // (n_cores * 4))
chunks = [(df_all.iloc[i:i+chunk_size]['ligand'].tolist(),
           df_all.iloc[i:i+chunk_size]['ic50'].tolist())
          for i in range(0, len(df_all), chunk_size)]
print(f'{len(chunks)} chunks, {n_cores} cores')

t1 = time.time()
with Pool(n_cores) as pool:
    results = pool.map(process_chunk, chunks)
X_bind = np.concatenate([r[0] for r in results])
y_bind = np.concatenate([r[1] for r in results])
# Convert ln(IC50_nM) to pIC50 (-log10 M) - same convention as pKd
y_bind = 9.0 - y_bind / 2.302585092994046
print(f'Fingerprints: {X_bind.shape} ({time.time()-t1:.0f}s)')
print(f'y: mean={y_bind.mean():.3f}, std={y_bind.std():.3f}')
np.save(f'{BASE}/X_bind.npy', X_bind)
np.save(f'{BASE}/y_bind.npy', y_bind)

In [ ]:
# 3. Load Phase 2 + Deduplicate BindingDB
X_phase2 = np.load(f'{BASE}/phase2_X.npy')
y_phase2 = np.load(f'{BASE}/phase2_y.npy')
p2_hashes = set(hash(r.tobytes()) for r in X_phase2[:, :512])
print(f'Phase 2: {len(X_phase2)} entries, {len(p2_hashes)} unique molecules')

# Build 1032-dim Phase 2 features (982 fp + 50 pocket)
pocket_full = pickle.load(open(f'{BASE}/pocket_features_full.pkl', 'rb'))
pdb_ids = pickle.load(open(f'{BASE}/phase2_pdb_ids.pkl', 'rb'))
X_phase2_aug = np.zeros((len(X_phase2), 1032), dtype=np.float32)
X_phase2_aug[:, :982] = X_phase2.astype(np.float32)
for i, pid in enumerate(pdb_ids):
    if pid in pocket_full:
        X_phase2_aug[i, 982:] = pocket_full[pid]
print(f'Phase 2 augmented: {X_phase2_aug.shape}')

# Dedup BindingDB against Phase 2
new_mask = np.ones(len(X_bind), dtype=bool)
unique_h = set()
for i in range(len(X_bind)):
    h = hash(X_bind[i, :512].tobytes())
    if h in p2_hashes or h in unique_h:
        new_mask[i] = False
    else:
        unique_h.add(h)

X_bind_new = X_bind[new_mask]
y_bind_new = y_bind[new_mask]
print(f'BindingDB: {len(X_bind)} total, {len(X_bind_new)} new, {len(X_bind)-len(X_bind_new)} dups')
np.save(f'{BASE}/X_bind_new.npy', X_bind_new)
np.save(f'{BASE}/y_bind_new.npy', y_bind_new)

In [ ]:
# 4. Combine Phase 2 + BindingDB and train with GPU
# Zero-pad BindingDB to 1032 (no pocket features)
X_bind_aug = np.zeros((len(X_bind_new), 1032), dtype=np.float32)
X_bind_aug[:, :982] = X_bind_new.astype(np.float32)

X_train = np.concatenate([X_phase2_aug, X_bind_aug])
y_train = np.concatenate([y_phase2, y_bind_new])
print(f'Training: {X_train.shape}, y: {y_train.mean():.3f}+-{y_train.std():.3f}')

ss = StandardScaler()
X_s = ss.fit_transform(X_train)
sel = SelectKBest(f_regression, k=500)
X_sel = sel.fit_transform(X_s, y_train)

model = xgb.XGBRegressor(
    max_depth=12, n_estimators=2000, learning_rate=0.01,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=3, gamma=0.1,
    reg_alpha=0.5, reg_lambda=2.0,
    tree_method='gpu_hist', random_state=42, n_jobs=-1, verbosity=1
)

print('Training XGBoost GPU on 575K entries...')
t0 = time.time()
model.fit(X_sel, y_train)
print(f'Done: {time.time()-t0:.0f}s ({((time.time()-t0)/60):.1f} min)')

model_data = {'model': model, 'scaler': ss, 'selector': sel,
              'config': 'Phase2+BindingDB, k=500, t=2000, GPU',
              'n_samples': len(X_train)}
with open(f'{BASE}/model_bindingdb.pkl', 'wb') as f:
    pickle.dump(model_data, f)

In [ ]:
# 5. Evaluate on CASF-2016
casf_df = pd.read_csv(f'{BASE}/casf2016_reference.csv')
print(f'{len(casf_df)} CASF-2016 complexes')

true_vals, pred_vals = [], []
for _, row in casf_df.iterrows():
    if pd.isna(row['smiles']) or not row['smiles']: continue
    fp = compute_fp(row['smiles'])
    if fp is None: continue
    X_test = fp.astype(np.float32).reshape(1, -1)
    X_test_s = ss.transform(X_test)
    X_test_sel = sel.transform(X_test_s)
    pred = model.predict(X_test_sel)[0]
    true_vals.append(row['pkd_true'])
    pred_vals.append(pred)

t, p = np.array(true_vals), np.array(pred_vals)
r_p, _ = pearsonr(t, p)
r_s, _ = spearmanr(t, p)
print(f'\n=== CASF-2016 ({len(t)} complexes) ===')
print(f'Pearson R: {r_p:.4f}')
print(f'Spearman R: {r_s:.4f}')
print(f'RMSE: {np.sqrt(np.mean((t-p)**2)):.4f}')
print(f'MAE: {np.mean(np.abs(t-p)):.4f}')

In [ ]:
# 6. Summary
print('='*60)
print('RESULTS SUMMARY')
print('='*60)
print(f'Training: Phase 2 ({len(X_phase2)}) + BindingDB ({len(X_bind_new)} new) = {len(X_train)} total')
print(f'Model: XGBoost GPU, k=500, t=2000')
print(f'CASF-2016 Pearson R: {r_p:.4f}')
print(f'CASF-2016 Spearman R: {r_s:.4f}')
print()
print('Comparison:')
print(f'  Phase 5c (19K only):             R=0.731')
print(f'  Phase 2 + BindingDB (575K):      R={r_p:.4f}')
print()
if r_p > 0.731:
    print('>>> BindingDB IMPROVES over Phase 2 alone!')
else:
    diff = 0.731 - r_p
    print(f'>>> BindingDB adds {len(X_bind_new)} molecules but R drops {diff:.3f}')